# Create LZP Awards (Latvian Council of Science / Latvijas Zinātnes padome)

Creates awards for the **Latvian Council of Science (LZP)** from LZP's own project pages at
https://www.lzp.gov.lv/lv/projekti (one page per funded project, /lv/projekts/<slug>; Latvian
government unified web platform, Drupal; robots.txt allows /lv/). Method 5 (static HTML).
The national R&D information system NZDIS (sciencelatvia.gov.lv) was checked first for a bulk
export: its project catalogue is login-only, so it is not used.

Each project page carries a category badge -- FLPP (Fundamental and Applied Research Projects),
VPP (National Research Programme and its sub-programmes), Swiss-Latvian cooperation programme ...
-- and a free-text block with start/end date, **Projekta numurs** (e.g. `lzp-2020/1-0088`,
`VPP-ZM-Lauksaimniecība-2026/1-0004`), **Finansējums** (EUR), **Projekta vadītājs** (PI),
**Projektu īsteno** (implementing institution), science field, summary (Latvian) and call.

**Scope filter (script):** pages in the category "LZP dalība projektos" (LZP's own role as a
partner in EU/ERDF projects -- not LZP-funded awards) and programme-overview pages with neither a
project number nor a PI are excluded. All LZP-funded pathways are kept.

**Fields:** `funder_award_id` = project number normalised to the cited form (`lzp-2020/1-0088`:
ASCII hyphens, lower-case prefix) -- see the collapse cell; `amount` in **EUR**; title and
summary in Latvian (LZP publishes no English versions on these pages); PI name split in the script
(canonical split_name); affiliation = implementing institution, country LV.

**Prerequisites:** run `scripts/local/lzp_to_s3.py` (uploads
`s3://openalex-ingest/awards/lzp/lzp_projects.parquet`, §1.4 shrink guard).

**Funder (Path A, F4320*):** funder_id `4320323269` (Latvijas Zinātnes Padome, ror 02ar66p97,
doi 10.13039/501100005375), read from `openalex.funders.funders`.

**Priority:** `535` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.lzp_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/lzp/lzp_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_projects, COUNT(DISTINCT funder_award_id) as distinct_ids,
       SUM(CASE WHEN award_id_is_synthetic = 'True' THEN 1 ELSE 0 END) AS synthetic_ids
FROM openalex.awards.lzp_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.lzp_raw;

In [ ]:
%sql
SELECT categories, COUNT(*) AS n, ROUND(SUM(TRY_CAST(amount AS DOUBLE)), 0) AS eur
FROM openalex.awards.lzp_raw
GROUP BY 1 ORDER BY n DESC;

## Step 1.6: Funder existence check (Path A)
F4320323269 must resolve to exactly one row in `openalex.funders.funders`; if not, STOP.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320323269;

## Step 2: Create LZP Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.lzp_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320323269  -- Latvian Council of Science
),
g AS (
    SELECT
        TRIM(funder_award_id) AS funder_award_id,
        NULLIF(TRIM(title_lv), '') AS title,
        NULLIF(TRIM(summary_lv), '') AS summary,
        TRY_CAST(amount AS DOUBLE) AS amount,
        COALESCE(NULLIF(TRIM(currency), ''), 'EUR') AS currency,
        NULLIF(TRIM(categories), '') AS scheme,
        TRY_TO_DATE(start_date, 'yyyy-MM-dd') AS start_d,
        TRY_TO_DATE(end_date, 'yyyy-MM-dd') AS end_d,
        CASE WHEN pi_family_name IS NOT NULL THEN named_struct(
            'given_name', NULLIF(TRIM(pi_given_name), ''),
            'family_name', NULLIF(TRIM(pi_family_name), ''),
            'orcid', CAST(NULL AS STRING),
            'role_start', CAST(NULL AS DATE),
            'affiliation', named_struct(
                'name', NULLIF(TRIM(institution), ''),
                'country', 'LV',
                'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
            )
        ) END AS lead,
        url
    FROM openalex.awards.lzp_raw
    WHERE funder_award_id IS NOT NULL AND TRIM(funder_award_id) != ''
)
SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000 as id,
    g.title as display_name,
    g.summary as description,
    f.funder_id,
    g.funder_award_id,
    g.amount,
    g.currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    'research' as funding_type,
    g.scheme as funder_scheme,
    'lzp_projects' as provenance,
    g.start_d as start_date,
    g.end_d as end_date,
    YEAR(g.start_d) as start_year,
    YEAR(g.end_d) as end_year,
    g.lead as lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
    CASE WHEN g.lead IS NOT NULL THEN array(g.lead) END as investigators,
    g.url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM g
CROSS JOIN src_funder f;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'lzp_projects' AND priority = 535;

-- Insert into openalex_awards_raw with priority 535 (direct funder ingest; higher wins
-- under the 2026-06-20 DESC dedup, so it outranks the priority-0 acknowledgement shells).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    535 as priority
FROM openalex.awards.lzp_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_awards, COUNT(DISTINCT id) as distinct_ids
FROM openalex.awards.lzp_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, amount, currency, start_date, end_date,
       lead_investigator.given_name, lead_investigator.family_name, lead_investigator.affiliation.name
FROM openalex.awards.lzp_awards LIMIT 20;

In [ ]:
%sql
SELECT funder_scheme, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_eur
FROM openalex.awards.lzp_awards
GROUP BY funder_scheme ORDER BY cnt DESC;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_eur
FROM openalex.awards.lzp_awards
GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- Section 6.4a frequency checks: PI and title top-20 must be a real long tail.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.lzp_awards
WHERE lead_investigator IS NOT NULL
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.lzp_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.7 coverage.
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(DISTINCT currency) as distinct_currencies,
    ROUND(MIN(amount), 0) as min_amt,
    ROUND(MAX(amount), 0) as max_amt,
    ROUND(AVG(amount), 0) as avg_amt,
    COUNT(start_date) as has_start_date,
    COUNT(end_date) as has_end_date,
    COUNT(lead_investigator) as has_pi
FROM openalex.awards.lzp_awards;

In [ ]:
%sql
-- Shape check (2.1.1): LZP / VPP project numbers; synthetic LZP-<slug> keys only for pages
-- that publish no number.
SELECT
    SUM(CASE WHEN funder_award_id RLIKE '^lzp-[0-9]{4}/[0-9]-[0-9]{3,5}$' THEN 1 ELSE 0 END) AS lzp_numbers,
    SUM(CASE WHEN funder_award_id RLIKE '^VPP-' THEN 1 ELSE 0 END) AS vpp_numbers,
    SUM(CASE WHEN funder_award_id LIKE 'LZP-%' THEN 1 ELSE 0 END) AS synthetic_keys,
    COUNT(*) AS total
FROM openalex.awards.lzp_awards;

In [ ]:
%sql
-- Citation stubs (priority < 3) under F4320323269 that collapse onto this ingest.
SELECT c.provenance, COUNT(*) as stub_rows, COUNT(t.id) as collapse_onto_lzp
FROM openalex.awards.openalex_awards_raw c
LEFT JOIN openalex.awards.lzp_awards t ON t.id = c.id
WHERE c.funder_id = 4320323269 AND c.priority < 3
GROUP BY c.provenance;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'lzp_projects'
GROUP BY provenance, priority;